## 课程实验

# D02：观察存储和写入批次

Data Warehousing with Apache Doris · Level 1

[讲义](course.md) · [课程入口](../README.md)


## 实验范围

仅重建 d02_batch 和 d02_small。小样本用于解释观测方法，不证明性能提升；后台 Compaction 可能很快消除版本差异。


In [ ]:
from dw_course import WarehouseLab
from dw_course.runtime import COURSE_ROOT, fixture, expect, normalized
from dw_course.schema import ORDER_COLUMNS, order_ddl, order_rows

lab = WarehouseLab()
print("Course database:", lab.database)



## 1. 对齐变量

两张表使用完全相同的 Schema、数据和桶数。关闭会合并小事务的 Group Commit，再比较一次批量写入和逐行写入。


In [ ]:
lab.execute("DROP TABLE IF EXISTS d02_batch")
ddl = order_ddl("d02_batch")
print(ddl)
lab.execute(ddl)
lab.execute("DROP TABLE IF EXISTS d02_small")
ddl = order_ddl("d02_small")
print(ddl)
lab.execute(ddl)
rows = order_rows(fixture("orders.json"))
lab.execute("SET group_commit = 'off_mode'")
lab.insert("d02_batch", ORDER_COLUMNS, rows)
for row in rows:
    lab.insert("d02_small", ORDER_COLUMNS, [row])


## 2. 查询元数据

SHOW TABLETS 中的 VersionCount 是版本相关观测，不等于直接数 Rowset。记录采样时刻、事务次数和后台合并影响。Rowset/Segment 深入检查需要受控管理接口，首版不自动调用。


In [ ]:
print(lab.query("SHOW CREATE TABLE d02_batch"))
print(lab.query("SHOW PARTITIONS FROM d02_batch"))
print(lab.query("SHOW TABLETS FROM d02_batch"))
print(lab.query("SHOW TABLETS FROM d02_small"))


## 3. 先确认业务结果没有变化

两张表都应为 10 行、1400.00。不把 VersionCount 必然更高或运行更快写成断言。


In [ ]:
for table in ("d02_batch", "d02_small"):
    expect(lab.query(f"SELECT COUNT(*), SUM(order_amount) FROM {table}"), [(10, "1400.00")])
print(lab.query("EXPLAIN SELECT order_id, order_amount FROM d02_batch WHERE order_id = 1001"))
lab.close()


## 待补的录制实验

放大数据的扫描量对照、Query Profile 展示、Rowset 管理接口和持续版本积压演示尚未实现；不能由这个小样本推导吞吐结论。
